# DTU student data export

Fetches the student view from AIT's SQL Server and writes it to `static_data.dtu_student_data`
in our PostgreSQL database, replacing the previous run's table.

Configured entirely through environment variables (or a `.env` file), so it can run unattended.
The connections come from `get_ait_engine()` and `get_pg_engine()` in `pis_lib`:

| Variable | Required | Default |
|---|---|---|
| `AIT_USERNAME` | yes | |
| `AIT_PASSWORD` | yes | |
| `AIT_DATABASE_TABLE` | no | `View_DTUPythonSupport_GetStudentData` |
| `PGUSER` | no | the account running the notebook |
| `PGPASSWORD` | only off the server | |

The AIT server and database are set in `pis_lib/connect.py`. On the server, PostgreSQL needs
no password: it logs in as the account running the notebook.

Run headless with:

```bash
uv run --with nbconvert jupyter nbconvert --to notebook --execute get_student_data.ipynb --output run.ipynb
```

In [ ]:
import logging
import os

import pandas as pd

from pis_lib.connect import get_ait_engine, get_pg_engine

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
_logger = logging.getLogger("get_student_data")

### Configuration

In [ ]:
DATABASE_TABLE = os.getenv("AIT_DATABASE_TABLE", "View_DTUPythonSupport_GetStudentData")
PG_SCHEMA = "static_data"
PG_TABLE = "dtu_student_data"

In [ ]:
ait = get_ait_engine()
pg = get_pg_engine()

### Fetch data

In [ ]:
def fetch(query: str) -> pd.DataFrame:
    _logger.info("Executing: %s", query)
    with ait.connect() as conn:
        result = conn.exec_driver_sql(query)
        # object dtype keeps the values as the driver returns them: None stays None, and
        # integer columns with NULLs don't turn into floats
        return pd.DataFrame(result.fetchall(), columns=list(result.keys()), dtype=object)

In [ ]:
data = fetch(f"SELECT * FROM {DATABASE_TABLE}")
_logger.info("Fetched %d rows", len(data))

if data.empty:
    raise RuntimeError(f"Query returned no rows from {DATABASE_TABLE}")

### Write to PostgreSQL

In [ ]:
with pg.begin() as conn:
    data.to_sql(PG_TABLE, conn, schema=PG_SCHEMA, if_exists="replace", index=False)

_logger.info("Wrote %d rows to %s.%s", len(data), PG_SCHEMA, PG_TABLE)